# WasteAI — TACO YOLO Training (Google Colab)

Trains a **28-class** waste detector for the WasteAI FYP.

## Before you start (on your PC)
1. Build zip (already done if you have `yolo_taco_colab.zip`).
2. Split for Drive upload:
```bat
.\.venv\Scripts\python.exe datasets\taco\split_colab_zip.py
```
3. Upload **ALL** files from `datasets/yolo_taco_colab_parts/` to Drive folder:
   `MyDrive/WasteAI/parts/`
   (every `yolo_taco_colab.zip.partXXX` + `MANIFEST.json`)

## In Colab
1. **Runtime → Change runtime type → GPU** (T4 / A100).
2. Run all cells in order (joins parts, verifies checksum, trains).
3. Download `best.pt` from Drive when training ends.

This does **not** change your local app until you manually replace `weights/best.pt`.
Nothing is dropped — parts reassemble to the full original zip.

## 1) Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → GPU'

## 2) Install Ultralytics

In [ ]:
!pip -q install ultralytics==8.3.0
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

## 3) Mount Google Drive, join split parts, unzip

Upload all parts to: `MyDrive/WasteAI/parts/`  
(`yolo_taco_colab.zip.part001`, `part002`, … + `MANIFEST.json`)

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import shutil
import json
import hashlib

drive.mount('/content/drive')

# >>> Folder that contains the part files + MANIFEST.json <<<
PARTS_DIR = Path('/content/drive/MyDrive/WasteAI/parts')
WORK = Path('/content/wasteai')
DATA_ROOT = WORK / 'yolo_taco'
JOINED_ZIP = WORK / 'yolo_taco_colab.zip'

WORK.mkdir(parents=True, exist_ok=True)
assert PARTS_DIR.exists(), f'Missing parts folder: {PARTS_DIR}'

manifest_path = PARTS_DIR / 'MANIFEST.json'
assert manifest_path.exists(), f'Missing {manifest_path}'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
parts = sorted(manifest['parts'], key=lambda p: p['name'])
print(f'Joining {len(parts)} parts (expected {manifest["original_bytes"]} bytes)...')

if JOINED_ZIP.exists():
    JOINED_ZIP.unlink()

with JOINED_ZIP.open('wb') as out:
    for p in parts:
        path = PARTS_DIR / p['name']
        assert path.exists(), f'Missing part: {path}'
        data = path.read_bytes()
        assert len(data) == p['bytes'], f'Size mismatch {p["name"]}'
        out.write(data)
        print(' +', p['name'], f'({len(data)/(1024**2):.1f} MiB)')

got = JOINED_ZIP.stat().st_size
assert got == manifest['original_bytes'], f'Size mismatch after join: {got} vs {manifest["original_bytes"]}'

print('Verifying SHA256 (a few minutes)...')
h = hashlib.sha256()
with JOINED_ZIP.open('rb') as f:
    while True:
        b = f.read(8 * 1024 * 1024)
        if not b:
            break
        h.update(b)
digest = h.hexdigest()
assert digest == manifest['original_sha256'], f'Checksum mismatch:\n got {digest}\n exp {manifest["original_sha256"]}'
print('Checksum OK — full archive restored, nothing lost.')

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

print('Unzipping...')
with zipfile.ZipFile(JOINED_ZIP, 'r') as zf:
    zf.extractall(WORK)

assert (DATA_ROOT / 'data.yaml').exists(), 'data.yaml not found after unzip'
train_n = len(list((DATA_ROOT / 'images' / 'train').glob('*')))
val_n = len(list((DATA_ROOT / 'images' / 'val').glob('*')))
print(f'Ready: train={train_n} val={val_n}')
print('DATA_ROOT =', DATA_ROOT)

## 4) Fix `data.yaml` paths for Colab

In [ ]:
import yaml

yaml_path = DATA_ROOT / 'data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = str(DATA_ROOT)
cfg['train'] = 'images/train'
cfg['val'] = 'images/val'

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(open(yaml_path, encoding='utf-8').read())

## 5) Train

- Default: train from **`yolov8s.pt`** (recommended on GPU).
- Optional: set `WEIGHTS` to a path of your old `best.pt` if you upload it.
- Outputs go to `/content/wasteai/runs/detect/taco_colab/` and are copied to Drive.

In [ ]:
from ultralytics import YOLO

# Recommended for Colab GPU:
WEIGHTS = 'yolov8s.pt'
# Or fine-tune your previous model after uploading it, e.g.:
# WEIGHTS = '/content/drive/MyDrive/WasteAI/best_old.pt'

EPOCHS = 50
IMGSZ = 640
BATCH = 16  # lower to 8 if you hit CUDA OOM
NAME = 'taco_colab'

model = YOLO(WEIGHTS)
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(WORK / 'runs' / 'detect'),
    name=NAME,
    exist_ok=True,
    workers=2,
    patience=20,
    plots=True,
)

print('Training finished.')
print(results)

## 6) Show metrics & copy weights to Drive

In [ ]:
from pathlib import Path
import shutil

run_dir = WORK / 'runs' / 'detect' / NAME
best = run_dir / 'weights' / 'best.pt'
last = run_dir / 'weights' / 'last.pt'
results_csv = run_dir / 'results.csv'

print('best exists:', best.exists(), best)
print('last exists:', last.exists(), last)

if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    # Ultralytics may prefix column names with spaces
    df.columns = [c.strip() for c in df.columns]
    map_col = 'metrics/mAP50(B)' if 'metrics/mAP50(B)' in df.columns else None
    if map_col:
        best_row = df.loc[df[map_col].idxmax()]
        print('Best epoch mAP50:', float(best_row[map_col]))
        print(best_row[['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']])

drive_out = Path('/content/drive/MyDrive/WasteAI/colab_outputs')
drive_out.mkdir(parents=True, exist_ok=True)

if best.exists():
    shutil.copy2(best, drive_out / 'best_taco_colab.pt')
    print('Copied to', drive_out / 'best_taco_colab.pt')
if results_csv.exists():
    shutil.copy2(results_csv, drive_out / 'results.csv')

# Optional: also offer a direct browser download
from google.colab import files
if best.exists():
    files.download(str(best))

## 7) Quick predict smoke test (optional)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best = Path('/content/wasteai/runs/detect/taco_colab/weights/best.pt')
sample = next((DATA_ROOT / 'images' / 'val').glob('*'))
print('Sample:', sample)

model = YOLO(str(best))
pred = model.predict(source=str(sample), conf=0.25, save=True)
print(pred[0].boxes)
print('Saved under runs/detect/predict*')

## After Colab — on your PC

1. Download `best_taco_colab.pt` from Drive.
2. Backup local weights:
   `copy weights\best.pt weights\best_backup.pt`
3. Only after you test photos, replace:
   `copy best_taco_colab.pt weights\best.pt`
4. Update `settings.py` `CLASSES` to the **28-class** list (required if you swap this model).

Ask the agent in Cursor to do the swap + settings sync when you are ready.